In [10]:
import pandas as pd
from pathlib import Path

### Conference Data


In [22]:
# Load the dataset
src_path = Path("../../data/processed/dataset_clean.csv")
df = pd.read_csv(src_path)

print("Rows, cols:", df.shape)
print("Columns:", list(df.columns))

Rows, cols: (3531, 23)
Columns: ['Conference', 'Year', 'Title', 'DOI', 'PaperType', 'Abstract', 'AuthorNames-Deduped', 'AuthorAffiliation', 'InternalReferences', 'AuthorKeywords', 'AminerCitationCount', 'CitationCount_CrossRef', 'PubsCited_CrossRef', 'Downloads_Xplore', 'Award', 'GraphicsReplicabilityStamp', 'MacroId', 'Cluster', 'MacroCategory', 'MacroSlug', 'ClusterShort', 'ClusterSlug', 'ClusterFileSlug']


In [19]:
# Clean the 'Conference' column
df["Conference"] = df["Conference"].astype("string").str.strip()

# Compute counts per conference type
counts = df["Conference"] \
        .value_counts(dropna=False) \
        .rename_axis("Conference") \
        .reset_index(name="Count")

# Save to CSV
out_path = Path("data/conferenceData.csv")
counts.to_csv(out_path, index=False)

### Topic Data

In [23]:
src_path = Path("../../data/processed/dataset_clean.csv")
df = pd.read_csv(src_path)

In [24]:
# Clean the 'MacroCategory' column
df["MacroCategory"] = df["MacroCategory"].astype("string").str.strip()

# Compute counts per macro category
counts = df["MacroCategory"] \
        .value_counts(dropna=False) \
        .rename_axis("MacroCategory") \
        .reset_index(name="Count")

# Save to CSV
out_path = Path("topicData.csv")
counts.to_csv(out_path, index=False)

### Year Data

In [25]:
src_path = Path("../../data/processed/dataset_clean.csv")
df = pd.read_csv(src_path)

In [26]:
# Ensure 'Year' is numeric (if it's already numeric, this does nothing harmful)
df["Year"] = pd.to_numeric(df["Year"], errors="coerce")

# Compute counts per year
counts = df["Year"] \
        .value_counts(dropna=False) \
        .rename_axis("Year") \
        .reset_index(name="Count")

# Sort by year (ascending)
counts = counts.sort_values("Year")

# Save to CSV
out_path = Path("yearData.csv")
counts.to_csv(out_path, index=False)

### Country Data

In [17]:

df = pd.read_csv("dataset_with_countries.csv")

# Build a stable paper identifier (prefer DOI, fallback to Title+Year)
df["paper_id"] = df["DOI"].fillna("").astype(str).str.strip()
fallback_id = df["Title"].astype(str).str.strip() + "_" + df["Year"].astype(str)
df.loc[df["paper_id"].eq(""), "paper_id"] = fallback_id[df["paper_id"].eq("")]

def parse_unique_countries(cell):
  """
  Convert a Country_Extracted cell into a unique list of countries for ONE paper.
  Handles separators:
    - ';' between authors
    - ',' sometimes used inside a token like 'Netherlands, Germany'
  """
  if pd.isna(cell):
    return []

  # First split by ';' (typical separator in your column)
  chunks = [c.strip() for c in str(cell).split(";") if c.strip()]

  countries = []
  for chunk in chunks:
    # If a chunk contains comma-separated countries, split those too
    parts = [p.strip() for p in chunk.split(",") if p.strip()]
    countries.extend(parts)

  # Dedupe within the same paper
  return sorted(set(countries))

df["countries_list"] = df["Country_Extracted"].apply(parse_unique_countries)

# One row per (paper, country)
exploded = df[["paper_id", "countries_list"]].explode("countries_list")
exploded = exploded.dropna(subset=["countries_list"]).rename(columns={"countries_list": "Country"})

# Count unique papers per country
country_counts = (
  exploded.groupby("Country")["paper_id"]
  .nunique()
  .reset_index(name="Count")
  .sort_values("Count", ascending=False)
)

# Save
out_path = Path("countryData.csv")
country_counts.to_csv(out_path, index=False)

print(country_counts.head(20))
print(f"Saved to: {out_path.resolve()}")


           Country  Count
47   United States   2394
14         Germany    594
29            None    316
7            China    295
5           Canada    238
2          Austria    204
46  United Kingdom    187
13          France    177
27     Netherlands    141
16       Hong Kong    123
41     Switzerland     83
1        Australia     59
40          Sweden     58
38     South Korea     51
30          Norway     47
22          Israel     40
4           Brazil     38
23           Italy     32
24           Japan     32
34    Saudi Arabia     30
Saved to: /Users/irynasavchuk/Desktop/DATAVIZ_PROJECT/DV/dv_repo/site/dataset/countryData.csv
